In [141]:
!pip install -q groq

In [142]:
from google.colab import drive
import sys
import torch


In [143]:
drive.mount('/content/drive')
sys.path.append(
"/content/drive/MyDrive/SupportAI"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [144]:
!ls -la /content/drive/MyDrive/SupportAI/src

total 10
-rw------- 1 root root 2256 Jul 10 17:57 inference.py
-rw------- 1 root root    0 Jul 10 17:39 __init__.py
-rw------- 1 root root 2870 Jul 10 18:24 llm.py
drwx------ 2 root root 4096 Jul 10 17:49 __pycache__


In [145]:
import sys

sys.path.insert(0, "/content/drive/MyDrive/SupportAI/src")

In [146]:
from inference import predict_all
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [147]:
path = "/content/drive/MyDrive/SupportAI/models"

In [148]:
def load_models(path):

    tokenizer = AutoTokenizer.from_pretrained(
        "DeepPavlov/rubert-base-cased"
    )

    models = {
        "topic": AutoModelForSequenceClassification.from_pretrained(
            f"{path}/topic"
        ),
        "sentiment": AutoModelForSequenceClassification.from_pretrained(
            f"{path}/sentiment"
        ),
        "intent": AutoModelForSequenceClassification.from_pretrained(
            f"{path}/intent"
        ),
    }

    return tokenizer, models

In [149]:
from groq import Groq

In [ ]:
API_KEY = "Ваш API ключ"

client = Groq(api_key=API_KEY)

In [151]:
def build_prompt(text, prediction):
    return f"""
Ты — опытный специалист службы поддержки клиентов.
Ниже представлены результаты автоматического анализа обращения.
Тема обращения:
{prediction["topic"]}
Тональность:
{prediction["sentiment"]}
Намерение клиента:
{prediction["intent"]}
Исходное сообщение клиента:
{text}
Твоя задача:

1. Ответить профессионально и вежливо.
2. Учитывать тему, тональность, срочность и намерение клиента.
3. Не придумывать информацию, которой нет.
4. Если данных недостаточно — попросить уточнение.
5. Предложить дальнейшие действия.
6. Ответ должен быть на русском языке.
"""

In [152]:
def generate_answer(text, models, tokenizer):
    prediction = predict_all(text, models, tokenizer)

    prompt = build_prompt(text, prediction)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.4
    )

    return response.choices[0].message.content

In [153]:
tokenizer, models = load_models(path)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [154]:
while True:
    text = input("\nВведите обращение (exit для выхода): ")

    if text.lower() == "exit":
        break

    answer = generate_answer(text, models, tokenizer)

    print("\nОтвет:\n")
    print(answer)


Введите обращение (exit для выхода): Почему так долго?

Ответ:

Здравствуйте! Благодарим вас за обращение в нашу службу поддержки. Мы понимаем, что вы обеспокоены тем, что что-то происходит слишком долго, и хотим помочь вам найти решение.

Поскольку ваше сообщение связано с оплатой, мы хотим уточнить, о какой именно проблеме вы говорите. Не могли бы вы предоставить нам больше информации о том, что именно происходит слишком долго? Это связано с обработкой платежа, доставкой или, может быть, с чем-то другим?

Мы ценим ваше терпение и готовы помочь вам решить эту проблему как можно скорее. Если у вас есть какие-либо дополнительные детали или вопросы, пожалуйста, не стесняйтесь делиться ими с нами. Мы здесь, чтобы помочь.

После получения более подробной информации мы сможем предложить вам наиболее подходящее решение или дальнейшие действия для решения вашей проблемы. Спасибо за сотрудничество и терпение. Мы ждем вашего ответа.

Введите обращение (exit для выхода): exit
